<a href="https://colab.research.google.com/github/asufarms-hue/explainable-ml-vegetation-drought/blob/main/01_xgboost_fundamentals.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import pandas as pd

data = {
    "rainfall": [120, 50, 95, 40, 130, 45, 100, 35, 115, 55],
    "temperature": [27, 34, 29, 35, 26, 33, 28, 36, 27, 32],
    "NDVI": [0.75, 0.35, 0.65, 0.30, 0.78, 0.38, 0.68, 0.25, 0.72, 0.40],
    "previous_NDVI": [0.73, 0.52, 0.68, 0.55, 0.76, 0.56, 0.70, 0.50, 0.74, 0.58],
    "drought": [0.10, 0.80, 0.30, 0.90, 0.05, 0.75, 0.20, 0.95, 0.15, 0.70],
    "stress": [0, 1, 0, 1, 0, 1, 0, 1, 0, 1]
}

df = pd.DataFrame(data)

df

,rainfall,temperature,NDVI,previous_NDVI,drought,stress
0,120,27,0.75,0.73,0.10,0
1,50,34,0.35,0.52,0.80,1
2,95,29,0.65,0.68,0.30,0
3,40,35,0.30,0.55,0.90,1
4,130,26,0.78,0.76,0.05,0
5,45,33,0.38,0.56,0.75,1
6,100,28,0.68,0.70,0.20,0
7,35,36,0.25,0.50,0.95,1
8,115,27,0.72,0.74,0.15,0
9,55,32,0.40,0.58,0.70,1


## Separate X and y

In [15]:
X = df[
    [
        "rainfall",
        "temperature",
        "NDVI",
        "previous_NDVI",
        "drought"
    ]
]

y = df["stress"]

## Check the contents in both x & y

In [16]:
X

,rainfall,temperature,NDVI,previous_NDVI,drought
0,120,27,0.75,0.73,0.10
1,50,34,0.35,0.52,0.80
2,95,29,0.65,0.68,0.30
3,40,35,0.30,0.55,0.90
4,130,26,0.78,0.76,0.05
5,45,33,0.38,0.56,0.75
6,100,28,0.68,0.70,0.20
7,35,36,0.25,0.50,0.95
8,115,27,0.72,0.74,0.15
9,55,32,0.40,0.58,0.70


In [17]:
y

,stress
0,0
1,1
2,0
3,1
4,0
5,1
6,0
7,1
8,0
9,1


In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [19]:
print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 8
Testing samples: 2


In [20]:
print("X_train:")
print(X_train)

print("\nX_test:")
print(X_test)

X_train:
   rainfall  temperature  NDVI  previous_NDVI  drought
5        45           33  0.38           0.56     0.75
0       120           27  0.75           0.73     0.10
7        35           36  0.25           0.50     0.95
2        95           29  0.65           0.68     0.30
9        55           32  0.40           0.58     0.70
4       130           26  0.78           0.76     0.05
3        40           35  0.30           0.55     0.90
6       100           28  0.68           0.70     0.20

X_test:
   rainfall  temperature  NDVI  previous_NDVI  drought
8       115           27  0.72           0.74     0.15
1        50           34  0.35           0.52     0.80


## XGBoost

In [21]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100, # estimated numbers of trees i.e how many boosting rounds/trees are used
    max_depth=3,      # This controls how deep each individual tree can become.
    learning_rate=0.1, # How strongly should each new tree contribute to improving the existing model?
    random_state=42
)

In [22]:
model.fit(X_train, y_train)   # Learn the relationship between these environmental features and the vegetation-stress labels.

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=100,
              n_jobs=None, num_parallel_tree=None, ...)

## Make predictions

In [23]:
y_pred = model.predict(X_test)

print("Predicted:", y_pred)
print("Actual:   ", y_test.values)

Predicted: [0 1]
Actual:    [0 1]


## how confident was XGBoost?

In [24]:
y_probability = model.predict_proba(X_test)

print(y_probability)

[[0.52497923 0.4750208 ]
 [0.47502083 0.5249792 ]]


## ACCURACY, PRECISION, RECALL & ROC-AUC

In [25]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score
)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1-score:", f1_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nROC-AUC:", roc_auc_score(y_test, y_probability[:, 1]))

Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0

Confusion Matrix:
[[1 0]
 [0 1]]

ROC-AUC: 1.0
